# Notebook 06 — Train Price Predictor → Export to .pkl

Trains a Random Forest / XGBoost price prediction model on Zimbabwe crop price history and exports it via joblib for production sklearn inference.

In [ ]:
import os, json, hashlib, time
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

EXPORTS_DIR = Path(os.getenv('EXPORTS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\exports\models'))
PRICE_CSV = Path(os.getenv('PRICE_DATASET_CSV', r'C:\Users\MJ\Desktop\Agric\jupyter\data\prices\price_history.csv'))
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v1')
ALLOW_SYNTHETIC_DATA = os.getenv('ALLOW_SYNTHETIC_DATA', 'false').lower() == 'true'
MIN_ROWS = int(os.getenv('MIN_PRICE_ROWS', '100'))
TEST_SIZE = float(os.getenv('PRICE_TEST_SIZE', '0.2'))

EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print('Price predictor configuration')
print('PRICE_CSV:', PRICE_CSV)
print('EXPORTS_DIR:', EXPORTS_DIR)
print('ALLOW_SYNTHETIC_DATA:', ALLOW_SYNTHETIC_DATA)

In [ ]:
required_columns = {'date', 'crop_type', 'quantity_kg', 'price_usd'}

if PRICE_CSV.exists():
    df = pd.read_csv(PRICE_CSV, parse_dates=['date'])
    dataset_source = str(PRICE_CSV)
else:
    if not ALLOW_SYNTHETIC_DATA:
        raise FileNotFoundError(f'Price CSV not found: {PRICE_CSV}. Set PRICE_DATASET_CSV or ALLOW_SYNTHETIC_DATA=true for smoke tests only.')
    print('No price CSV found — generating synthetic data for smoke-testing only')
    rng = np.random.default_rng(42)
    n = 5000
    crops = ['maize', 'tomato', 'soya_beans', 'groundnuts', 'cabbage']
    df = pd.DataFrame({
        'date': pd.date_range('2020-01-01', periods=n, freq='h'),
        'crop_type': rng.choice(crops, n),
        'quantity_kg': rng.uniform(10, 5000, n),
        'price_usd': rng.uniform(0.10, 2.50, n),
        'market': rng.choice(['Harare', 'Bulawayo', 'Mutare'], n),
    })
    dataset_source = 'synthetic_smoke_test'

missing_cols = sorted(required_columns - set(df.columns))
if missing_cols:
    raise ValueError(f'Price dataset missing required columns: {missing_cols}')
if len(df) < MIN_ROWS:
    raise ValueError(f'Price dataset has {len(df)} rows, minimum required is {MIN_ROWS}')

print('Dataset source:', dataset_source)
print('Dataset shape:', df.shape)
df.head(3)

In [ ]:
CROP_INDEX = {c: i for i, c in enumerate(['maize', 'mango', 'tomato', 'soya_beans', 'groundnuts',
                                           'tobacco', 'cotton', 'cabbage', 'potato', 'onion',
                                           'sugar_beans', 'sunflower'])}

df['crop_idx']      = df['crop_type'].map(CROP_INDEX).fillna(0).astype(int)
df['log_quantity']  = np.log1p(df['quantity_kg'])
df['day_of_week']   = pd.to_datetime(df['date']).dt.dayofweek
df['month']         = pd.to_datetime(df['date']).dt.month
df['days_ahead']    = 7

FEATURES = ['quantity_kg', 'days_ahead', 'crop_idx', 'log_quantity', 'day_of_week', 'month']
TARGET   = 'price_usd'

df = df.dropna(subset=FEATURES + [TARGET])
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)
print('Feature matrix shape:', X.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

model = RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
model.fit(X_train_s, y_train)

preds = model.predict(X_val_s)
mae  = mean_absolute_error(y_val, preds)
r2   = r2_score(y_val, preds)
print(f'Val MAE: {mae:.4f} USD/kg | R²: {r2:.4f}')

In [ ]:
model_path = EXPORTS_DIR / 'price_predictor_v1.pkl'
scaler_path = EXPORTS_DIR / 'price_scaler_v1.pkl'

joblib.dump(model, model_path, compress=3)
joblib.dump(scaler, scaler_path, compress=3)
print('Saved:', model_path, scaler_path)

meta = {
    'model': 'price_predictor_v1',
    'version': MODEL_VERSION,
    'format': 'pkl',
    'source_notebook': '06_train_price_predictor.ipynb',
    'dataset_source': dataset_source,
    'dataset_rows': int(len(df)),
    'features': FEATURES,
    'target': TARGET,
    'crop_index': CROP_INDEX,
    'val_mae': round(float(mae), 4),
    'val_r2': round(float(r2), 4),
    'sha256': sha256_file(model_path),
    'scaler_sha256': sha256_file(scaler_path),
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
metadata_path = EXPORTS_DIR / 'price_predictor_metadata.json'
metadata_path.write_text(json.dumps(meta, indent=2))
print('Metadata:', json.dumps(meta, indent=2))

In [ ]:
m2 = joblib.load(model_path)
s2 = joblib.load(scaler_path)
sample = np.array([[100.0, 7.0, 0.0, np.log1p(100), 1, 5]], dtype=np.float32)
pred = m2.predict(s2.transform(sample))[0]
print(f'Smoke-test price prediction: {pred:.4f} USD/kg — price_predictor_v1.pkl ready for production.')